# Full-data re-modelling — 5-HT2A & Dopamine D3

Runs the **complete** modelling pipeline on the verified full fetch. No ChEMBL needed — it only processes files you already have.

**Put these three files in the working folder:**
`ki_verified_5-HT2A.csv`, `ki_verified_Dopamine_D3.csv`, and `curated_15desc.csv` (the locked v36 set; `data/curated_15desc.csv` also works).

**Expected set sizes (already confirmed): 5-HT2A = 4,620, D3 = 5,488.**
Runtime: ~20-30 min (the y-randomisation is the slow part). When it's done, **copy the whole output of the last two cells back to the chat** and I'll update the thesis.

In [ ]:
!pip -q install rdkit descriptastorus xgboost scikit-learn pandas numpy

In [ ]:
import os, numpy as np, pandas as pd, warnings, time; warnings.filterwarnings('ignore')
from rdkit import Chem, DataStructs, RDLogger; RDLogger.DisableLog('rdApp.*')
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem.Scaffolds import MurckoScaffold
from descriptastorus.descriptors import rdNormalizedDescriptors
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, root_mean_squared_error as rmse
from xgboost import XGBRegressor

LOCKED = 'data/curated_15desc.csv' if os.path.exists('data/curated_15desc.csv') else 'curated_15desc.csv'
FILES  = {'5-HT2A': 'ki_verified_5-HT2A.csv', 'Dopamine D3': 'ki_verified_Dopamine_D3.csv'}
KEY    = {'5-HT2A': '5-HT2A', 'Dopamine D3': 'D3'}
for f in [LOCKED, *FILES.values()]:
    assert os.path.exists(f), f'MISSING FILE: {f}'
print('all input files found')

In [ ]:
# ---- curation + build the modelling sets (same pipeline as the thesis) ----
sr = SaltRemover(); pep = Chem.MolFromSmarts('[NX3][CX4][CX3](=O)[NX3][CX4][CX3](=O)')
def parent(s):
    m = Chem.MolFromSmiles(str(s)) if isinstance(s, str) else None
    if m is None: return None
    p = sr.StripMol(m, dontRemoveEverything=True)
    if p is None or p.GetNumAtoms() == 0:
        fr = Chem.GetMolFrags(m, asMols=True); p = max(fr, key=lambda x: x.GetNumAtoms()) if fr else m
    return p
def landrum(g):
    v = g['pKi'].round(2); v = v[~v.duplicated(keep='first')]
    v = [x for x in v if not any(abs(x-o-3.0) < 1e-6 or abs(x-o+3.0) < 1e-6 for o in v)]
    return max(v) if v else np.nan
locked = pd.read_csv(LOCKED)
def build(name):
    raw = pd.read_csv(FILES[name]); raw['standard_value'] = pd.to_numeric(raw['standard_value'], errors='coerce')
    raw = raw[raw['standard_value'] > 0].copy(); raw['pKi'] = 9.0 - np.log10(raw['standard_value'])
    raw['mol'] = raw['canonical_smiles'].apply(parent); raw = raw[raw['mol'].notna()]
    raw = raw[raw['mol'].apply(lambda m: Descriptors.MolWt(m)) < 700]
    raw = raw[~raw['mol'].apply(lambda m: m.HasSubstructMatch(pep))]
    raw['parent_smiles'] = raw['mol'].apply(Chem.MolToSmiles)
    agg = raw.groupby('molecule_chembl_id').apply(
            lambda g: pd.Series({'pKi': landrum(g), 'parent_smiles': g['parent_smiles'].iloc[0]})
          ).reset_index().dropna(subset=['pKi'])
    ex = locked[locked.target.str.contains(KEY[name], case=False)][['molecule_chembl_id','pKi','parent_smiles']].drop_duplicates('molecule_chembl_id')
    pr = agg[~agg.molecule_chembl_id.isin(ex.molecule_chembl_id)]
    both = pd.concat([ex, pr], ignore_index=True).drop_duplicates('molecule_chembl_id').reset_index(drop=True)
    print(f'{name}: {len(ex)} clinical + {len(pr)} new = {len(both)} compounds')
    return both
sets = {n: build(n) for n in FILES}

In [ ]:
# ---- featurise: ECFP4 (2048) + DescriptaStorus RDKit2DNormalized (~200) = fused ----
def featurize(dfset):
    gen = rdNormalizedDescriptors.RDKit2DNormalized()
    mols = [Chem.MolFromSmiles(s) for s in dfset.parent_smiles]; keep = [i for i, m in enumerate(mols) if m]
    dfset = dfset.iloc[keep].reset_index(drop=True); mols = [mols[i] for i in keep]; N = len(mols)
    E = np.zeros((N, 2048), np.float32)
    for i, m in enumerate(mols):
        DataStructs.ConvertToNumpyArray(AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048), E[i])
    D = np.nan_to_num(np.array([gen.process(s)[1:] for s in dfset.parent_smiles], dtype=np.float32))
    scaf = np.array([Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(m)) for m in mols])
    return np.hstack([E, D]), dfset.pKi.values.astype(float), scaf
feat = {}
for n in sets:
    t0 = time.time(); feat[n] = featurize(sets[n]); print(f'{n} featurised: X{feat[n][0].shape}  [{time.time()-t0:.0f}s]')

In [ ]:
# ---- CELL A: XGBoost & RF, scaffold split (12 seeds) + 10-fold CV ----
def boost(): return XGBRegressor(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8,
                                 colsample_bytree=0.6, random_state=0, n_jobs=-1)
def rf():    return RandomForestRegressor(n_estimators=400, random_state=0, n_jobs=-1)
MODELS = {'XGB': boost, 'RF': rf}
def scaffold_split(scaf, N, seed, tf=0.2):
    rng = np.random.RandomState(seed); g = {}
    for i, s in enumerate(scaf): g.setdefault(s, []).append(i)
    ks = list(g); rng.shuffle(ks); nt = int(round(tf*N)); te = []
    for k in ks:
        if len(te) >= nt: break
        te += g[k]
    te = set(te); return [i for i in range(N) if i not in te], sorted(te)
rows = []
for t in feat:
    X, y, scaf = feat[t]; N = len(y)
    for mn, mk in MODELS.items():
        rs = []
        for s in range(12):
            tr, te = scaffold_split(scaf, N, s)
            if len(te) < 5 or len(np.unique(y[te])) < 2: continue
            rs.append(r2_score(y[te], mk().fit(X[tr], y[tr]).predict(X[te])))
        rs = np.array(rs)
        pred = np.zeros(N)
        for tr, te in KFold(10, shuffle=True, random_state=0).split(X):
            pred[te] = mk().fit(X[tr], y[tr]).predict(X[te])
        rows.append({'target': t, 'model': mn, 'n': N,
                     'scaffold_R2': round(rs.mean(), 3),
                     'scaffold_CI': f'[{np.percentile(rs,2.5):.2f},{np.percentile(rs,97.5):.2f}]',
                     'CV10_R2': round(r2_score(y, pred), 3),
                     'CV10_RMSE': round(rmse(y, pred), 3)})
        print('done:', t, mn)
res = pd.DataFrame(rows); print('\n===== TABLE A: scaffold R2 / CI / 10-fold CV ====='); print(res.to_string(index=False))

In [ ]:
# ---- CELL B: representation, y-randomisation, learning curve, read-across ----
def _sm(X, y, scaf, seeds=12):
    rs = []
    for s in range(seeds):
        tr, te = scaffold_split(scaf, len(y), s)
        if len(te) < 5 or len(np.unique(y[te])) < 2: continue
        rs.append(r2_score(y[te], boost().fit(X[tr], y[tr]).predict(X[te])))
    return float(np.mean(rs))
print('===== SUPPORTING ANALYSES =====')
print('1) REPRESENTATION (XGB scaffold R2, mean of 12)')
for t in feat:
    X, y, scaf = feat[t]
    for lab, Xr in [('ECFP4', X[:, :2048]), ('DescriptaStorus', X[:, 2048:]), ('Fused', X)]:
        print(f'   {t:12s} {lab:15s} {_sm(Xr, y, scaf):.3f}')
print('2) Y-RANDOMISATION (100 permutations, scaffold split seed 0)')
for t in feat:
    X, y, scaf = feat[t]; tr, te = scaffold_split(scaf, len(y), 0)
    true = r2_score(y[te], boost().fit(X[tr], y[tr]).predict(X[te])); rng = np.random.RandomState(0)
    null = np.array([r2_score(y[te], boost().fit(X[tr], rng.permutation(y[tr])).predict(X[te])) for _ in range(100)])
    print(f'   {t:12s} true={true:.3f}  null_mean={null.mean():.3f}  max={null.max():.3f}  p={(int((null>=true).sum())+1)/101:.4f}')
print('3) LEARNING CURVE (XGB, scaffold split seed 0)')
for t in feat:
    X, y, scaf = feat[t]; tr, te = scaffold_split(scaf, len(y), 0); tr = np.array(tr); np.random.RandomState(0).shuffle(tr)
    for frac in [0.15, 0.3, 0.5, 0.75, 1.0]:
        k = max(40, int(len(tr)*frac))
        print(f'   {t:12s} n_train={k:5d}  R2={r2_score(y[te], boost().fit(X[tr[:k]], y[tr[:k]]).predict(X[te])):.3f}')
print('4) READ-ACROSS (1-NN Tanimoto baseline, mean of 12)')
for t in feat:
    X, y, scaf = feat[t]; B = (X[:, :2048] > 0).astype(np.uint8); pc = B.sum(1); rs = []
    for s in range(12):
        tr, te = scaffold_split(scaf, len(y), s); tr = np.array(tr)
        pred = [y[tr[int(np.argmax((B[tr]&B[i]).sum(1)/np.maximum(1, pc[tr]+pc[i]-(B[tr]&B[i]).sum(1))))]] for i in te]
        if len(np.unique(y[te])) > 1: rs.append(r2_score(y[te], np.array(pred)))
    print(f'   {t:12s} read-across R2={np.mean(rs):.3f}')
print('\nDONE — copy the output of CELL A and CELL B back to the chat.')